In [1]:
from fastai.vision.all import *

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
path = Path('/kaggle/input/cassava-leaf-disease-classification/')
path.ls()

(#17) [Path('/kaggle/input/cassava-leaf-disease-classification/train.csv.zip'),Path('/kaggle/input/cassava-leaf-disease-classification/sample_submission.csv.zip'),Path('/kaggle/input/cassava-leaf-disease-classification/test.zip'),Path('/kaggle/input/cassava-leaf-disease-classification/test_tfrecords.zip'),Path('/kaggle/input/cassava-leaf-disease-classification/train_tfrecords.zip'),Path('/kaggle/input/cassava-leaf-disease-classification/test_images.zip'),Path('/kaggle/input/cassava-leaf-disease-classification/train.csv'),Path('/kaggle/input/cassava-leaf-disease-classification/test_images'),Path('/kaggle/input/cassava-leaf-disease-classification/test_tfrecords'),Path('/kaggle/input/cassava-leaf-disease-classification/cassava-leaf-disease-classification'),Path('/kaggle/input/cassava-leaf-disease-classification/train_tfrecords'),Path('/kaggle/input/cassava-leaf-disease-classification/train_images'),Path('/kaggle/input/cassava-leaf-disease-classification/train.zip'),Path('/kaggle/input/cas

In [3]:
train_df = pd.read_csv(path/'train.csv')
train_df.head()
with open(path/'label_num_to_disease_map.json') as f:
    label_dict = json.load(f)

label_dict = {int(k):v for k,v in label_dict.items()}
df = train_df.set_index('image_id')
labels = df.to_dict()['label']

def get_label(labels, x):
    x = Path(x)
    return labels[x.name]

def get_data(labels, bs=64, presize=500, resize=384):
    tfms = [Rotate(90), Warp(magnitude=0.4, p=1.),Zoom(min_zoom=0.9, max_zoom=1.3), Brightness(max_lighting=0.5), Flip(), Contrast(), Resize(resize)]
    comp = setup_aug_tfms(tfms)
    return DataBlock(blocks=(ImageBlock, CategoryBlock),
        get_items=lambda p: get_image_files(p),
        get_y=partial(get_label, labels),
        splitter=RandomSplitter(),
        item_tfms=[Resize(presize)],
        batch_tfms=[*comp, Normalize.from_stats(*imagenet_stats)]).dataloaders(path/'train_images',bs=bs,num_workers=8)

In [4]:
dls = get_data(labels,bs=128, presize=384)
learn = cnn_learner(dls, models.resnet50, metrics=[accuracy], pretrained=False)
test_files = get_image_files(path/'test_images')
predictions = []
for fold in range(1):
    learn.load(f'/kaggle/input/casava-starter/models/resnet50-full-fold_{fold}')
    test_dl = dls.test_dl(test_files)
    preds = learn.tta(dl=test_dl, n=6)
    predictions.append(preds[0])

/usr/local/lib/python3.11/dist-packages/fastai/vision/learner.py:303: UserWarning: `cnn_learner` has been renamed to `vision_learner` -- please update your code
  warn("`cnn_learner` has been renamed to `vision_learner` -- please update your code")


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/casava-starter/models/resnet50-full-fold_0.pth'

In [5]:
predictions

[]

In [6]:
preds = torch.argmax(torch.mean(torch.stack(predictions), dim=0), dim=1)
test_fn = map(lambda x: x.name, test_files)

RuntimeError: stack expects a non-empty TensorList

In [7]:
submission = pd.DataFrame({'image_id': test_fn, 'label': preds})
submission.to_csv('submission.csv', index=False)

NameError: name 'test_fn' is not defined

In [8]:
submission

NameError: name 'submission' is not defined